In [32]:
import pandas as pd
import numpy as np
df = pd.read_csv(
    "D:/ecommerce-customer-intelligence/data/raw/data.csv",
    encoding="latin1"
)



In [35]:
print("shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())
print("The duplicated data:", df.duplicated().sum())
print("\nDATA TYPES : " , df.dtypes)
print("\n Basic STATS: " , df.describe())


shape: (541909, 8)

Missing Values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
The duplicated data: 5268

DATA TYPES :  InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

 Basic STATS:              Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.690570
std       218.081158      96.759853    1713.600303
min    -80995.000000  -11062.060000   12346.000000
25%         1.000000       1.250000   13953.000000
50%         3.000000       2.080000   15152.000000
75%        10.000000       4.130000   16791.000000
max     80995.000000   38970.000000   18287.000000


    InvoiceNo StockCode                        Description  Quantity  \
517    536409     21866        UNION JACK FLAG LUGGAGE TAG         1   
527    536409     22866      HAND WARMER SCOTTY DOG DESIGN         1   
537    536409     22900    SET 2 TEA TOWELS I LOVE LONDON          1   
539    536409     22111       SCOTTIE DOG HOT WATER BOTTLE         1   
555    536412     22327  ROUND SNACK BOXES SET OF 4 SKULLS         1   
587    536412     22273               FELTCRAFT DOLL MOLLY         1   
589    536412     22749  FELTCRAFT PRINCESS CHARLOTTE DOLL         1   
594    536412     22141     CHRISTMAS CRAFT TREE TOP ANGEL         1   
598    536412     21448          12 DAISY PEGS IN WOOD BOX         1   
600    536412     22569        FELTCRAFT CUSHION BUTTERFLY         2   

         InvoiceDate  UnitPrice  CustomerID         Country  
517  12/1/2010 11:45       1.25     17908.0  United Kingdom  
527  12/1/2010 11:45       2.10     17908.0  United Kingdom  
537  12/1/2010 11:45 

In [39]:

# 135,080 missing = no identity = useless for churn
# We DROP because we cannot guess a customer's ID
before = df.shape[0]
df = df[df['CustomerID'].notna()]
after = df.shape[0]
print(f"Removed {before - after} rows with missing CustomerID")

Removed 0 rows with missing CustomerID


In [44]:
# Currently text — useless for time calculations
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df['InvoiceDate'].dtype)  # Should show datetime64
print(df['InvoiceDate'].min())  # Earliest transaction
print(df['InvoiceDate'].max())  # Latest transaction


datetime64[ns]
2010-12-01 08:26:00
2011-12-09 12:50:00


In [45]:
# Currently float (12345.0) — convert to clean integer
df['CustomerID'] = df['CustomerID'].astype(int)
print(df['CustomerID'].dtype)  # Should show int64

int64


In [46]:
# Keep as string — it's an ID not a number
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

In [51]:
# Remove Negative and Zero Quantity
print("Quantity Before")
print(df['Quantity'].describe())

df = df[df['Quantity'] > 0]
print("\nQuantity stats AFTER:")
print(df['Quantity'].describe())

Quantity Before
count    406829.000000
mean         12.061303
std         248.693370
min      -80995.000000
25%           2.000000
50%           5.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

Quantity stats AFTER:
count    397924.000000
mean         13.021823
std         180.420210
min           1.000000
25%           2.000000
50%           6.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64


In [52]:
#Remove Zero and Negative UnitPrice

print("UnitPrice stats BEFORE:")
print(df['UnitPrice'].describe())

df = df[df['UnitPrice'] > 0]

print("\nUnitPrice stats AFTER:")
print(df['UnitPrice'].describe())
# Min should now be above 0

UnitPrice stats BEFORE:
count    397924.000000
mean          3.116174
std          22.096788
min           0.000000
25%           1.250000
50%           1.950000
75%           3.750000
max        8142.750000
Name: UnitPrice, dtype: float64

UnitPrice stats AFTER:
count    397884.000000
mean          3.116488
std          22.097877
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
max        8142.750000
Name: UnitPrice, dtype: float64


In [53]:
# Remove Cancelled Invoices

cancelled = df[df['InvoiceNo'].str.startswith('C')].shape[0]
print(f"Cancelled invoices found: {cancelled}")

df = df[~df['InvoiceNo'].str.startswith('C')]
print(f"Remaining rows: {df.shape[0]}")

Cancelled invoices found: 0
Remaining rows: 397884


In [54]:
# Check for extreme outliers in Quantity and UnitPrice
print("Quantity outliers:")
Q1 = df['Quantity'].quantile(0.25)
Q3 = df['Quantity'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 3 * IQR
print(f"Upper limit: {upper_limit}")
print(f"Rows above limit: {df[df['Quantity'] > upper_limit].shape[0]}")

# For now just flag them — don't remove yet
df['Quantity_Outlier'] = df['Quantity'] > upper_limit

Quantity outliers:
Upper limit: 42.0
Rows above limit: 18527


In [55]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Verify
print(df[['Quantity', 'UnitPrice', 'TotalPrice']].head())
print("\nTotalPrice stats:")
print(df['TotalPrice'].describe())
print(f"\nAny zero or negative TotalPrice: {df[df['TotalPrice'] <= 0].shape[0]}")

   Quantity  UnitPrice  TotalPrice
0         6       2.55       15.30
1         6       3.39       20.34
2         8       2.75       22.00
3         6       3.39       20.34
4         6       3.39       20.34

TotalPrice stats:
count    397884.000000
mean         22.397000
std         309.071041
min           0.001000
25%           4.680000
50%          11.800000
75%          19.800000
max      168469.600000
Name: TotalPrice, dtype: float64

Any zero or negative TotalPrice: 0


In [60]:
# Save return behavior BEFORE we lost that data
#  returns might be  a churn signal)
returns_df = df[df['Quantity'] < 0].copy()
returns_df.to_csv('D:/ecommerce-customer-intelligence/data/processed/returns_data.csv', index=False)
print(f"Returns saved: {returns_df.shape[0]} rows")

Returns saved: 0 rows


In [57]:
print("="*40)
print("FINAL DATA HEALTH CHECK")
print("="*40)
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['CustomerID'].nunique()}")
print(f"Unique countries: {df['Country'].nunique()}")

FINAL DATA HEALTH CHECK
Total rows: 397884
Total columns: 10

Missing values:
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID          0
Country             0
Quantity_Outlier    0
TotalPrice          0
dtype: int64

Data types:
InvoiceNo                   object
StockCode                   object
Description                 object
Quantity                     int64
InvoiceDate         datetime64[ns]
UnitPrice                  float64
CustomerID                   int64
Country                     object
Quantity_Outlier              bool
TotalPrice                 float64
dtype: object

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Unique customers: 4338
Unique countries: 37


In [59]:
df.to_csv('D:/ecommerce-customer-intelligence/data/processed/ecommerce_clean.csv', index=False)

print("Clean data saved successfully!")

Clean data saved successfully!
